In [1]:
!pip install fastapi uvicorn pyngrok

In [2]:
import os
import sklearn
import pandas as pd
from joblib import load
import json

In [3]:
!ngrok authtoken 2tQR3hMKgY1918rkKxTdGfu9g5s_4S59sDie9wVDY55tEjvZa

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [4]:
model = load('heart_lr_model.pkl')

In [5]:
model.feature_names_in_

array(['sbp', 'tobacco', 'ldl', 'adiposity', 'famhist', 'typea',
       'obesity', 'alcohol', 'age'], dtype=object)

In [6]:
%%writefile app.py

import os
import pandas as pd
import numpy as np
from joblib import load
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI()

# Load the trained model
ml_model = load('heart_lr_model.pkl')

# Define input data schema
class PredictionRequest(BaseModel):
    sbp: int
    tobacco: float
    ldl: float
    adiposity: float
    famhist: str  # This is categorical
    typea: int
    obesity: float
    alcohol: float
    age: int

@app.post("/predict")
def predict(input_data: PredictionRequest):
    try:
        # Convert input to DataFrame
        df = pd.DataFrame([input_data.dict()])

        # Predict using the model
        prediction = ml_model.predict(df)[0]

        # Return the result
        return {"chd_prediction": int(prediction)}

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction error: {e}")

Overwriting app.py


In [7]:
!nohup uvicorn app:app --host 0.0.0.0 --port 6010 &

nohup: appending output to 'nohup.out'


In [8]:
!ps -ax | grep uvicorn

   9650 ?        Rl     0:00 /usr/bin/python3 /usr/local/bin/uvicorn app:app --host 0.0.0.0 --port 6
   9652 ?        S      0:00 /bin/bash -c ps -ax | grep uvicorn
   9654 ?        S      0:00 grep uvicorn


In [12]:
from pyngrok import ngrok

# Expose the FastAPI app
public_url = ngrok.connect(6010)
print(f"Public URL: {public_url}")

Public URL: NgrokTunnel: "https://becf-35-243-136-90.ngrok-free.app" -> "http://localhost:6010"


## Alert!

Run the following commands only at the end, to stop the ngrok and uvicorn service


In [11]:
ngrok.kill()

In [11]:
!kill -9 <pid of uvicorn service>

/bin/bash: -c: line 1: syntax error near unexpected token `newline'
/bin/bash: -c: line 1: `kill -9 <pid of uvicorn service>'
